# Read H-Reflex App Data Files

This notebook reads and visualizes data from the **H-Reflex Behavior App** (hreflex_txbdc) binary data files.

**File convention (V2/V3):**
- **`.hrs1`** — MH Recruitment Curve stage: sweeps across stimulation intensities to map the M/H-wave recruitment curve.
- **`.hrs2`** — Control Mode stage: stimulates at a fixed user-set intensity (can be changed between trials).
- **`.hrs3`** — Down Condition Pellet (DCP) stage: closed-loop H-reflex conditioning with pellet reward.
- **`.hrs4`** — Up Condition Pellet stage.
- **`.hrs5`** — Down Condition VNS stage.
- **`.hrs6`** — Up Condition VNS stage.
- **`.hrsft`** — Frequency Test stage.

All trial files share the MhRecHeader + MhRecTrial binary format.
EMG data blocks (raw differential, filtered, abs-value) are embedded in every file.

The binary format is based on the `FileIO_Helpers` serialization from the `hreflex_txbdc` package.

# Section 1: Binary File Reader Utilities

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (
    # File readers
    read_hrs2, read_hrs3, read_hrs4, read_hrs5, read_hrs6, read_hrs_ft,
    find_hrs_files, detect_app_version,
    # Summary printers
    print_hrs2_summary,
    # HRS2 plots
    plot_amplitude_distribution, plot_background_emg_views, plot_hrs2_analysis,
    plot_actual_trial_timeline, plot_mwave_control_error, plot_frequency_test,
    # Frequency Test analysis plots
    plot_ft_depression_curve, plot_ft_averaged_waveforms, plot_ft_peak_curve,
    # SNR analysis
    compute_snr_analysis, compute_mra_snr_analysis,
    plot_hrs2_trials, classify_trials, get_trial_window,
    detect_stim_onset, get_trial_context_window,
    detect_and_correct_failed_trials,
    # Post-hoc global windowing analysis
    analyze_global_background, run_threshold_sweep, plot_threshold_sweep,
    split_trials_by_polarity, plot_hm_ratio_summary, plot_hwave_regression,
    # Constants
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
    build_merged_amp_groups,
    make_viewer, compute_h_comparison_data, plot_h_reflex_comparison,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")

# Section 1b: Auto-Detect Recording Files

Set `recording_dir` to the path of your recording folder.  
The `.hrs1` and `.hrs2` files will be found automatically.

In [5]:
# ── Multi-Recording Configuration ─────────────────────────────────────────────
# Each entry: ("Display Label", "relative/path/to/recording_dir", sample_rate_hz)
# sample_rate_hz: explicit Hz (e.g. 10000.0 or 5000.0); None = auto-detect from data.
RECORDING_DIRS = [

    #("HRPILOT-17 CC1",  "HRPilot-17_Control/CC1_HRPILOT-17_BOOTH1_500US_6-30-26",        15000.0),
    #("HRPILOT-17 CC2",  "HRPilot-17_Control/CC2_HRPILOT-17_BOOTH1_500US_7-2-26",         15000.0),
    #("HRPILOT-17 CC3",  "HRPilot-17_Control/CC3_HR-PILOT-17_BOOTH1_500US_7-3-26",  15000.0),
    #("HRPILOT-17 CC4",  "HRPilot-17_Control/CC4_HR-PILOT-17_BOOTH1_500US_7-6-26",  15000.0),
    #("HRPILOT-17 CC5",  "HRPilot-17_Control/CC5_HRPILOT-17_BOOTH1_500US_7-8-26",  15000.0),
    #("HRPILOT-17 CC6",  "HRPilot-17_Control/CC6_HRPILOT-17_BOOTH1_500US_7-10-26",  15000.0),
    
    #("HRPILOT-17 CTRL1",  "HRPilot-17_Control/HRPILOT-17_M-WAVE_ALGORITHM_TEST_BOOTH1_7-14-26",        15000.0),
    #("HRPILOT-17 CTRL2",  "HRPilot-17_Control/HRPILOT-17_CONTROL_BOOTH1_250US_7-16-26",         15000.0),
    #("HRPILOT-17 CTRL3",  "HRPilot-17_Control/HRPILOT-17_CONTROL2_BOOTH1_250US_10KHZ_7-17-26",  10000.0),
    #("HRPILOT-17 CTRL4",  "HRPilot-17_Control/HRPILOT-17_CONTROL3_BOOTH1_250US_10KHZ_7-21-26",  10000.0),
    #("HRPILOT-17 CTRL5",  "HRPilot-17_Control/HRPILOT-17_CONTROL4_BOOTH1_250US_10KHZ_7-22-26",  10000.0),
    
    
    ("HRPILOT-17 Test11",  "HRPilot-17_Control/CCC1_HRPILOT-17_BOOTH1_10KHZ_250US_8-5-26",  10000.0),
    ("HRPILOT-17 ControlM1",  "HRPilot-17_Control/CCC2_HRPILOT-17_BOOTH1_10KHZ_250US_8-6-26",  10000.0),
    ("HRPILOT-17 ControlM2",  "HRPilot-17_Control/CCC3_HRPILOT-17_BOOTH1_10KHZ_250US_8-7-26",  10000.0),
    #("HRPILOT-17 M4",  "HRPilot-17_Control/CCC4_HRPILOT-17_BOOTH1_10KHZ_250US_8-11-26",  10000.0),
    ("HRPILOT-17 ControlM3",  "HRPilot-17_Control/CCC5_HRPILOT-17_BOOTH1_10KHZ_250US_8-12-26",  10000.0),
    
    
    #("HRPILOT-18 CC1",  "HRPilot-18_Control/CC1_HRPILOT-18_BOOTH1_500US_6-29-26",        15000.0),
    #("HRPILOT-18 CC2",  "HRPilot-18_Control/HRPILOT-18_CONTROL1_BOOTH2_250US_10KHZ_7-17-26",         15000.0),
    #("HRPILOT-18 CC3",  "HRPilot-18_Control/HRPILOT-18_CONTROL2_BOOTH1_500US_7-20-26",  15000.0),
    #("HRPILOT-18 CC4",  "HRPilot-18_Control/HRPILOT-18_CONTROL3_BOOTH2_250US_10KHZ_7-21-26",  15000.0),
    #("HRPILOT-18 CC5",  "HRPilot-18_Control/HRPILOT-18_CONTROL4_BOOTH2_250US_10KHZ_7-22-26",  15000.0),
    #("HRPILOT-18 CC6",  "HRPilot-18_Control/HRPILOT-18_CONTROL4_BOOTH2_250US_10KHZ_7-22-26",  15000.0),
        
    
    
    
    #("HRPILOT-18 CTRL1",  "HRPilot-18_Control/HRPILOT-18_M-WAVE_ALGORITHM_TEST_BOOTH1_7-16-26",        15000.0),
    #("HRPILOT-18 CTRL2",  "HRPilot-18_Control/HRPILOT-18_CONTROL1_BOOTH2_250US_10KHZ_7-17-26",         10000.0),
    #("HRPILOT-18 CTRL3",  "HRPilot-18_Control/HRPILOT-18_CONTROL2_BOOTH1_500US_7-20-26",  10000.0),
    #("HRPILOT-18 CTRL4",  "HRPilot-18_Control/HRPILOT-18_CONTROL3_BOOTH2_250US_10KHZ_7-21-26",  10000.0),
    #("HRPILOT-18 CTRL5",  "HRPilot-18_Control/HRPILOT-18_CONTROL4_BOOTH2_250US_10KHZ_7-22-26",  10000.0),
    
    
    
    
    #("HRPILOT-21 21",  "HRPILOT-21_TEST1_BOOTH1_250US_7-17-26",        10000.0),
    #("HRPILOT-24 CCC1",  "HRPILOT-24_CALIBRATION1_BOOTH1_250US_7-21-26",        10000.0),
    #("HRPILOT-24 CCC1",  "HRpilot-17_Control/CCC1_HRPILOT-17_BOOTH1_10KHZ_250US_8-5-26",        10000.0),
    #("HRPILOT-24 CCC2",  "HRPILOT-24_CALIBRATION2_BOOTH2_200US_7-22-26",        10000.0),
    
    # Add more recordings below — uncomment or append new tuples.
]
# When multiple recordings are loaded a Recording dropdown appears in the selector cell.
print(f"{len(RECORDING_DIRS)} recording(s) configured.")
for _i, (_lbl, _rdir, _rsr) in enumerate(RECORDING_DIRS):
    _sr_str = f"{_rsr} Hz" if _rsr else "auto-detect"
    print(f"  [{_i}] {_lbl!r}  →  {_rdir}  (sample_rate={_sr_str})")

4 recording(s) configured.
  [0] 'HRPILOT-17 Test11'  →  HRPilot-17_Control/CCC1_HRPILOT-17_BOOTH1_10KHZ_250US_8-5-26  (sample_rate=10000.0 Hz)
  [1] 'HRPILOT-17 ControlM1'  →  HRPilot-17_Control/CCC2_HRPILOT-17_BOOTH1_10KHZ_250US_8-6-26  (sample_rate=10000.0 Hz)
  [2] 'HRPILOT-17 ControlM2'  →  HRPilot-17_Control/CCC3_HRPILOT-17_BOOTH1_10KHZ_250US_8-7-26  (sample_rate=10000.0 Hz)
  [3] 'HRPILOT-17 ControlM3'  →  HRPilot-17_Control/CCC5_HRPILOT-17_BOOTH1_10KHZ_250US_8-12-26  (sample_rate=10000.0 Hz)


In [6]:
# ── Load all recordings (auto-detects V2 / V3) ─────────────────────────────────
_all_recordings = {}

for (_rlabel, _rdir, _rsr) in RECORDING_DIRS:
    print(f'\n── Loading: {_rlabel!r}  ({_rdir})')
    _rp1, _rp2, _rp3, _rp4, _rp5, _rp6, _rpft = find_hrs_files(_rdir)
    _rav = detect_app_version(_rdir)

    _r_cm_h  = _r_cm_t  = _r_cm_e  = None
    _r_dcp_h = _r_dcp_t = _r_dcp_e = None
    _r_s4_h  = _r_s4_t  = _r_s4_e  = None
    _r_s5_h  = _r_s5_t  = _r_s5_e  = None
    _r_s6_h  = _r_s6_t  = _r_s6_e  = None
    _r_ft_h  = _r_ft_t  = _r_ft_e  = None
    _r_h2h   = _r_h1h   = None
    _r_h2t   = _r_h2e   = []

    # V2/V3: .hrs1 = MH Recruitment Curve, .hrs2 = Control Mode, .hrs3+ = conditioning stages
    if _rp1:
        _r_h2h, _r_h2t, _r_h2e = read_hrs2(_rp1)
        _r_h1h = _r_h2h
        print(f'   .hrs1: {len(_r_h2t)} trials  (MH Recruitment)')
    else:
        print('   .hrs1: not found')
    if _rp2:
        _r_cm_h, _r_cm_t, _r_cm_e = read_hrs2(_rp2)
        print(f'   .hrs2: {len(_r_cm_t)} trials  (Control Mode)')
    else:
        print('   .hrs2: not found')
    if _rp3:
        _r_dcp_h, _r_dcp_t, _r_dcp_e = read_hrs3(_rp3)
        print(f'   .hrs3: {len(_r_dcp_t)} trials  (Down Condition Pellet)')
    if _rav >= 3:
        if _rp4:
            _r_s4_h, _r_s4_t, _r_s4_e = read_hrs4(_rp4)
            print(f'   .hrs4: {len(_r_s4_t)} trials  (Up Condition Pellet)')
        if _rp5:
            _r_s5_h, _r_s5_t, _r_s5_e = read_hrs5(_rp5)
            print(f'   .hrs5: {len(_r_s5_t)} trials  (Down Condition VNS)')
        if _rp6:
            _r_s6_h, _r_s6_t, _r_s6_e = read_hrs6(_rp6)
            print(f'   .hrs6: {len(_r_s6_t)} trials  (Up Condition VNS)')
        if _rpft:
            _r_ft_h, _r_ft_t, _r_ft_e = read_hrs_ft(_rpft)
            print(f'   .hrsft: {len(_r_ft_t)} trials  (Frequency Test)')

    # Control Mode-only: alias as primary analysis when no MH Recruitment stage
    if not _r_h2t and _r_cm_t:
        _r_h2h = _r_cm_h
        _r_h2t = _r_cm_t
        _r_h2e = _r_cm_e
        _r_h1h = _r_h2h
        print('   Note: Control Mode aliased as primary analysis (no MH Recruitment stage).')

    # Resolve sample rate
    _r_detect_sr = _rsr
    if _r_detect_sr is None:
        _r_detect_sr = getattr(_r_h1h, 'sample_rate', None) or 5000.0

    # hrs1_header fallback stub
    if _r_h1h is None:
        _r_sr_val = _r_detect_sr
        class _SampleRateStub:
            sample_rate = _r_sr_val
        _r_h1h = _SampleRateStub()

    # Build stage map for this recording
    _r_sm = {}
    if _r_h2t and (not _r_cm_t or _r_h2t is not _r_cm_t):
        _r_sm['mh_recruitment'] = (_r_h2t, _r_h2h, _r_h2e, 'MH Recruitment Curve (.hrs1)')
    if _r_cm_t:
        _r_sm['control_mode']   = (_r_cm_t,  _r_cm_h,  _r_cm_e,  'Control Mode (.hrs2)')
    if _r_dcp_t:
        _r_sm['dcp']            = (_r_dcp_t, _r_dcp_h, _r_dcp_e, 'Down Condition Pellet (.hrs3)')
    if _r_s4_t:
        _r_sm['up_cond_pellet'] = (_r_s4_t,  _r_s4_h,  _r_s4_e,  'Up Condition Pellet (.hrs4)')
    if _r_s5_t:
        _r_sm['down_cond_vns']  = (_r_s5_t,  _r_s5_h,  _r_s5_e,  'Down Condition VNS (.hrs5)')
    if _r_s6_t:
        _r_sm['up_cond_vns']    = (_r_s6_t,  _r_s6_h,  _r_s6_e,  'Up Condition VNS (.hrs6)')

    _all_recordings[_rlabel] = {
        'stage_map':    _r_sm,
        'sample_rate':  _r_detect_sr,
        'hrs1_header':  _r_h1h,
        'ft_trials':    _r_ft_t,
        'ft_header':    _r_ft_h,
        'app_version':  _rav,
    }
    print(f'   App V{_rav}  |  Stages: {list(_r_sm.keys())}  |  SR: {_r_detect_sr} Hz')

_active_rec_label = next(iter(_all_recordings))
print(f'\n{len(_all_recordings)} recording(s) loaded.  Active: {_active_rec_label!r}')


── Loading: 'HRPILOT-17 Test11'  (HRPilot-17_Control/CCC1_HRPILOT-17_BOOTH1_10KHZ_250US_8-5-26)
   .hrs1: not found
   .hrs2: 505 trials  (Control Mode)
   .hrsft: 37 trials  (Frequency Test)
   Note: Control Mode aliased as primary analysis (no MH Recruitment stage).
   App V3  |  Stages: ['control_mode']  |  SR: 10000.0 Hz

── Loading: 'HRPILOT-17 ControlM1'  (HRPilot-17_Control/CCC2_HRPILOT-17_BOOTH1_10KHZ_250US_8-6-26)
   .hrs1: 805 trials  (MH Recruitment)
   .hrs2: 899 trials  (Control Mode)
   .hrsft: 109 trials  (Frequency Test)
   App V3  |  Stages: ['mh_recruitment', 'control_mode']  |  SR: 10000.0 Hz

── Loading: 'HRPILOT-17 ControlM2'  (HRPilot-17_Control/CCC3_HRPILOT-17_BOOTH1_10KHZ_250US_8-7-26)
   .hrs1: not found
   .hrs2: 287 trials  (Control Mode)
   Note: Control Mode aliased as primary analysis (no MH Recruitment stage).
   App V3  |  Stages: ['control_mode']  |  SR: 10000.0 Hz

── Loading: 'HRPILOT-17 ControlM3'  (HRPilot-17_Control/CCC5_HRPILOT-17_BOOTH1_10KHZ_25

# Section 3: Peri-Stimulus Trials — MH Recruitment Curve or Control Mode

`hrs2_trials` contains whichever stage was run:
- `.hrs1` present → MH Recruitment Curve trials
- `.hrs1` absent, `.hrs2` present → Control Mode trials (aliased automatically)

In [7]:
# Stage summary for the active recording
print(f'Active recording: {_active_rec_label!r}')
_rec_info = _all_recordings[_active_rec_label]
for _sk, (_st, _sh, _se, _slbl) in _rec_info['stage_map'].items():
    print(f'  {_slbl}: {len(_st)} trials')
print(f'\n(Run the Stage Selector cell below to enable the interactive dropdown.)')

Active recording: 'HRPILOT-17 Test11'
  Control Mode (.hrs2): 505 trials

(Run the Stage Selector cell below to enable the interactive dropdown.)


In [8]:
# ── Recording & Stage Viewer Factory ───────────────────────────────────────────
# Each viewer section below has its own independent Recording + Stage dropdowns.
# Set each viewer to a different recording/stage to compare them side by side.
# make_viewer() is defined in helpers.py
from IPython.display import display as _disp

# ── Loaded recordings summary ────────────────────────────────────────────────────
print(f'{len(_all_recordings)} recording(s) loaded:')
for _rl, _rd in _all_recordings.items():
    print(f'  {_rl!r}  (App V{_rd["app_version"]}  |  {_rd["sample_rate"]} Hz)')
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        print(f'    · {_slbl}: {len(_st)} trials')
print()
print('Each viewer below has its own Recording + Stage dropdowns for independent selection.')

# ── Stimulation Intensity Histogram ─────────────────────────────────────────────────────
def _render_hist(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Histogram: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_amplitude_distribution(trials, header)

_hist_widget, _hist_render = make_viewer(_all_recordings, _active_rec_label, _render_hist)
_disp(_hist_widget)
_hist_render()

4 recording(s) loaded:
  'HRPILOT-17 Test11'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 505 trials
  'HRPILOT-17 ControlM1'  (App V3  |  10000.0 Hz)
    · MH Recruitment Curve (.hrs1): 805 trials
    · Control Mode (.hrs2): 899 trials
  'HRPILOT-17 ControlM2'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 287 trials
  'HRPILOT-17 ControlM3'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 782 trials

Each viewer below has its own Recording + Stage dropdowns for independent selection.


In [9]:
# ── Trial Timeline ─────────────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_tl(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Trial Timeline: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_actual_trial_timeline(trials, header=header)

_tl_widget, _tl_render = make_viewer(_all_recordings, _active_rec_label, _render_tl)
_disp(_tl_widget)
_tl_render()

# Section 3b: "Most Recent Background" + "Background EMG Level"

Recreates the H-Reflex App recruitment-curve trial-plot widgets from `MhRecruitmentCurveStage.get_trial_plot_options`:

- **Most recent background** bar chart of the pre-stim |EMG| bins.
- **EMG Level** scatter of background grand means across trials.

Bins are reconstructed from `hrs2_emg_blocks` over a fixed monitoring window (default 2500 ms ending at trigger time).

In [10]:
# ── Background EMG Views ──────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_bg(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Background EMG: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_background_emg_views(trials, emg_blocks, monitoring_window_ms=2500)

_bg_widget, _bg_render = make_viewer(_all_recordings, _active_rec_label, _render_bg)
_disp(_bg_widget)
_bg_render()

# Section 6: HRS2 Detailed Analysis

Interactive averaged-waveform paged grid (with M/H peak markers and signal overlays) plus the normalized and raw recruitment curves. The cell below sets the analysis parameters; the cell after that calls `plot_hrs2_analysis`.

In [11]:
#  Configuration 
PRE_PLOT_MS  = 5   # ms before stim onset to display
POST_PLOT_MS = 25  # ms after  stim onset to display
N_PER_PAGE   = 6   # trials shown per page (2 rows x 3 cols)

# M/H wave window constants (ms relative to stim onset)
#M_WAVE_START_MS = 3
#M_WAVE_END_MS= 5.5
#H_WAVE_START_MS = 9.0
#H_WAVE_END_MS   = 12

M_WAVE_START_MS = 2.2 
M_WAVE_END_MS= 4.2
H_WAVE_START_MS = 6   
H_WAVE_END_MS   = 9.6

PRE_AVG_MS  = 5   # ms before stim onset
POST_AVG_MS = 25  # ms after  stim onset
N_PER_PAGE  = 6   # amplitude groups per page (2 rows x 3 cols)


In [12]:
# ── Pre-compute Comparison Data ───────────────────────────────────────────────────────
# Computed once here (after configuration constants are set) for instant rendering
# in the comparison plot below. Re-run this cell if you change PRE_AVG_MS,
# POST_AVG_MS, M_WAVE_START_MS, M_WAVE_END_MS, H_WAVE_START_MS, or H_WAVE_END_MS.
_xr_cache = compute_h_comparison_data(
    _all_recordings, PRE_AVG_MS, POST_AVG_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
)
_xr_stages = {}
for _rd in _all_recordings.values():
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        if _st and _sk not in _xr_stages:
            _xr_stages[_sk] = _slbl
print(f'Comparison data pre-computed for {len(_xr_cache)} recording(s), '
      f'{len(_xr_stages)} stage(s): {list(_xr_stages.keys())}')

Comparison data pre-computed for 4 recording(s), 2 stage(s): ['control_mode', 'mh_recruitment']


In [13]:
# ── Optional: Merged Amplitude Group Analysis ─────────────────────────────────
# MERGE_ALL = True  →  collapse every amplitude into a single group.
#
# MERGED_GROUPS accepts two styles (mix freely):
#   Explicit list : [0.12, 0.13, 0.15]   — merge exactly those amplitudes
#   Range tuple   : (0.10, 0.50)         — merge all amplitudes where low <= amp <= high
#
# Examples:
#   MERGED_GROUPS = [[0.12, 0.13], [0.15, 0.16, 0.17]]   ← two explicit groups
#   MERGED_GROUPS = [(0.10, 0.20), (0.25, 0.40)]          ← two range groups
#   MERGED_GROUPS = [(0.10, 0.20), [0.50, 0.55]]          ← range + explicit, mixed
#
# Leave both flags at their defaults to use the standard (unmerged) grouping.
MERGE_ALL     = False   # True → collapse every amplitude into one group
MERGED_GROUPS = []
#[(0,0.108), (0.108,0.23), (0.23,0.43), (0.43,0.50), (0.50,0.60), (0.60,0.70)]

def _resolve_groups(trials, groups):
    _all_amps = sorted({t.stimulation_amplitude_ma for t in trials})
    resolved = []
    for g in groups:
        if isinstance(g, tuple) and len(g) == 2:
            lo, hi = float(g[0]), float(g[1])
            matched = [a for a in _all_amps if lo <= a <= hi]
            if matched:
                resolved.append(matched)
            else:
                print(f'Warning: range ({lo}, {hi}) matched no amplitudes — skipped.')
        else:
            resolved.append(list(g))
    return resolved

def _apply_merge(trials):
    if MERGE_ALL:
        _amps = sorted({t.stimulation_amplitude_ma for t in trials})
        return build_merged_amp_groups(trials, [_amps])
    if MERGED_GROUPS:
        return build_merged_amp_groups(trials, _resolve_groups(trials, MERGED_GROUPS))
    return trials

print("Merge config: MERGE_ALL =", MERGE_ALL, " | MERGED_GROUPS =", MERGED_GROUPS or "(none)")
print("Re-run viewer cells or change Recording/Stage to apply new merge settings.")

Merge config: MERGE_ALL = False  | MERGED_GROUPS = (none)
Re-run viewer cells or change Recording/Stage to apply new merge settings.


In [14]:
# ── HRS2 Analysis: Interactive Averaged Waveforms + Recruitment Curve ─────────
from IPython.display import display as _disp

def _render_ana(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    tp = _apply_merge(trials)
    print(f'\n── Analysis: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_hrs2_analysis(
        tp, header,
        pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        emg_blocks=emg_blocks,
    )

_ana_widget, _ana_render = make_viewer(_all_recordings, _active_rec_label, _render_ana)
_disp(_ana_widget)
_ana_render()

# Section 6c: Frequency Test Analysis (V3 FT)

Three views available via the **View** toggle:

- **H/M MRA Per Pulse** — mean ± 1σ H-wave and M-wave MRA at each pulse position across all trials. Uses `pulse_h_wave_mra` / `pulse_m_wave_mra` pre-stored by the app (mean rectified average within each window). The most-recent trial is overlaid as a dashed line. Shows homosynaptic (rate-dependent) depression across the train.
- **Avg Waveforms** — paged 2×3 grid. Each tile is one pulse position: individual trial segments are drawn at low alpha, the bold black trace is the cross-trial mean. M-wave window is blue-shaded; H-wave window is green-shaded (matching the HRS2 Analysis viewer). MRA annotations are boxed above each window. A colorbar at the top encodes pulse # (blue = pulse 1, red = last). Use the **Page** slider to page through pulses.
- **H/M Peak Per Pulse** — same structure as H/M MRA Per Pulse, but computes `max(|EMG|)` within each wave window directly from the raw EMG trace, then averages across trials.

M/H wave windows are shared with the HRS2 configuration constants (`M_WAVE_START_MS`, `H_WAVE_START_MS`, etc.).

In [ ]:
# ── Frequency Test Analysis: Interactive Viewer ────────────────────────────────
# View toggle:  H/M MRA Per Pulse | Avg Waveforms | H/M Peak Per Pulse
# Page slider (Avg Waveforms only): step through pulse positions 6 at a time.
# M/H wave windows use M_WAVE_START_MS / H_WAVE_START_MS from the config cell above.
from ipywidgets import Dropdown, ToggleButtons, IntSlider, Output, VBox
from IPython.display import display as _disp

_ft_recs = [rl for rl in _all_recordings if _all_recordings[rl].get('ft_trials')]
if not _ft_recs:
    print("No Frequency Test data loaded (.hrft not found in any recording directory).")
else:
    _ft_rec_d = Dropdown(
        options=_ft_recs, value=_ft_recs[0],
        description='Recording:', layout={'width': '600px'}
    )
    _ft_view_d = ToggleButtons(
        options=[
            ('H/M MRA Per Pulse', 'depression'),
            ('Avg Waveforms',     'waveforms'),
            ('H/M Peak Per Pulse','peak'),
        ],
        description='View:', style={'button_width': '185px'},
    )
    # Page slider — shown only for the Avg Waveforms view
    _ft_page_s = IntSlider(
        min=1, max=1, step=1, value=1,
        description='Page:', layout={'width': '450px', 'visibility': 'hidden'}
    )
    _ft_out = Output()

    def _ft_render():
        rl   = _ft_rec_d.value
        vw   = _ft_view_d.value
        rec  = _all_recordings[rl]
        ft_t = rec.get('ft_trials', [])
        ft_h = rec.get('ft_header')
        sr   = rec.get('sample_rate') or getattr(ft_h, 'sample_rate', None)
        if not ft_t or ft_h is None:
            return

        # Update page slider max to match actual pulse count / page size
        n_p  = getattr(ft_h, 'n_pulses_per_train', 0) or \
               max((len(getattr(t, 'pulse_h_wave_mra', [])) for t in ft_t), default=1)
        tot  = max(1, int(np.ceil(n_p / N_PER_PAGE)))
        _ft_page_s.max = tot
        _ft_page_s.layout.visibility = 'visible' if vw == 'waveforms' else 'hidden'

        hz = round(1e6 / ft_h.event_period_us, 1) if getattr(ft_h, 'event_period_us', 0) else '?'
        with _ft_out:
            _ft_out.clear_output(wait=True)
            print(f'Frequency Test: {len(ft_t)} trials  |  '
                  f'{n_p} pulses/train  |  '
                  f'{hz} Hz  [{rl}]')
            if vw == 'depression':
                plot_ft_depression_curve(ft_t, ft_h, sample_rate=sr)
            elif vw == 'waveforms':
                plot_ft_averaged_waveforms(
                    ft_t, ft_h,
                    pre_pulse_ms=2.0, post_pulse_ms=POST_PLOT_MS,
                    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
                    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
                    sample_rate=sr, n_per_page=N_PER_PAGE,
                    page=_ft_page_s.value - 1,
                )
            else:  # peak
                plot_ft_peak_curve(
                    ft_t, ft_h, sample_rate=sr,
                    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
                    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
                )

    def _ft_on_rec(c):
        ft_t = _all_recordings[_ft_rec_d.value].get('ft_trials', [])
        _ft_page_s.value = 1
        _ft_render()

    def _ft_on_page(c):
        if _ft_view_d.value == 'waveforms':
            _ft_render()

    _ft_rec_d.observe(_ft_on_rec,                   names='value')
    _ft_view_d.observe(lambda c: _ft_render(),       names='value')
    _ft_page_s.observe(_ft_on_page,                  names='value')

    _disp(VBox([_ft_rec_d, _ft_view_d, _ft_page_s, _ft_out]))
    _ft_render()

# Section 6b: H-Reflex Size Across Recordings

**H-reflex size per amplitude group** is the **Mean Rectified Amplitude (MRA) of the averaged bipolar waveform** in the H-wave window, minus the MRA of the pre-stimulus background:

> **size (µV) = mean|avg_bip(t ∈ [H_START, H_END])| − mean|avg_bip(t < 0)|**

Steps per amplitude group in each recording:
1. All trials at that amplitude are time-aligned and averaged → `avg_bip`
2. **H-wave MRA** = mean of |avg_bip| within [H_WAVE_START_MS, H_WAVE_END_MS]
3. **Background MRA** = mean of |avg_bip| in the pre-stimulus window (t < 0)
4. **H-reflex size** = H-wave MRA − background MRA

This matches the green `H: X.X µV` annotation shown in the HRS2 Analysis viewer above, with background subtracted.

The plot below shows one **box-and-whisker per recording** in `RECORDING_DIRS` order, with each amplitude group contributing one data point:
- **Box** — interquartile range (25th–75th percentile across amplitude groups)
- **Horizontal bar** — median
- **◆ diamond** — mean
- **Whiskers** — mean ± 1 standard deviation
- **n=** — number of amplitude groups in that recording for the selected stage
- **Dashed line** — connects per-recording means in `RECORDING_DIRS` order

In [16]:
# ── Cross-Recording Comparison Plot ──────────────────────────────────────────────────
# Toggle between H-Reflex size, M-Wave size, and Background MRA (pre-stim EMG level).
# Re-run the pre-compute cell above if you change analysis configuration constants.
from ipywidgets import Dropdown, ToggleButtons, Output, VBox
from IPython.display import display as _disp

_xr_stage_d = Dropdown(
    options=[(_slbl, _sk) for _sk, _slbl in _xr_stages.items()],
    description='Stage:', layout={'width': '480px'}
)
if _xr_stages:
    _xr_stage_d.value = next(iter(_xr_stages))

_xr_metric_d = ToggleButtons(
    options=[('H-Reflex Size', 'h_reflex'), ('M-Wave Size', 'm_wave'), ('Background MRA', 'background')],
    description='Metric:',
    style={'button_width': '160px'},
)

_xr_out = Output()

def _xr_render():
    sk = _xr_stage_d.value
    mt = _xr_metric_d.value
    if not sk:
        return
    with _xr_out:
        _xr_out.clear_output(wait=True)
        plot_h_reflex_comparison(_xr_cache, RECORDING_DIRS, sk, _xr_stages, metric=mt)

_xr_stage_d.observe(lambda c: _xr_render(), names='value')
_xr_metric_d.observe(lambda c: _xr_render(), names='value')
_disp(VBox([_xr_stage_d, _xr_metric_d, _xr_out]))
_xr_render()

# Section 3c: H:M Ratio Summary

Box plot and histogram of H:M ratio for each stimulation polarity group.
If both normal and reversed polarities were used in this session, each group is analysed separately.

In [17]:
# ── Stim Polarity Analysis ────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_pol(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Polarity / H:M Ratio: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    trials_by_polarity = split_trials_by_polarity(trials)
    plot_hm_ratio_summary(
        trials_by_polarity, header,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
    )
    plot_hwave_regression(
        trials, emg_blocks,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
    )

_pol_widget, _pol_render = make_viewer(_all_recordings, _active_rec_label, _render_pol)
_disp(_pol_widget)
_pol_render()

# Section 3d: M-Wave Stabilization Control Error (V3 S2 / S4 / S5 / S6)

Plots the M-wave stabilization controller output trial-by-trial.
- **Left axis** — `m_wave_error` (µV): difference between actual M-wave and the target set-point; falls back to `m_wave_window_median` when the controller was inactive.
- **Right axis** — `stimulation_amplitude_ma` (mA, orange): stimulation intensity actually used.

Available only for V3 recordings (file_version ≥ 9 for S2; file_version ≥ 2 for S4/S5/S6).

In [18]:
# ── M-Wave Control Error (V3 stages with closed-loop M-wave stabilization) ────
from IPython.display import display as _disp

def _mw_stage_filter(key, trials, header, emg_blocks, label):
    return bool(trials) and any(
        not (getattr(t, 'm_wave_error', float('nan')) != getattr(t, 'm_wave_error', float('nan')))
        or not (getattr(t, 'm_wave_window_median', float('nan')) != getattr(t, 'm_wave_window_median', float('nan')))
        for t in trials)

def _render_mw(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── M-Wave Control Error: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_mwave_control_error(trials, header)

_has_any_mw = any(
    any(_mw_stage_filter(sk, _t, _h, _e, lbl)
        for sk, (_t, _h, _e, lbl) in rd['stage_map'].items())
    for rd in _all_recordings.values()
)
if _has_any_mw:
    _mw_widget, _mw_render = make_viewer(_all_recordings, _active_rec_label, _render_mw, stage_filter=_mw_stage_filter)
    _disp(_mw_widget)
    _mw_render()
else:
    print("No M-wave stabilization data available in any loaded recording "
          "(requires V3 S2 file_version ≥ 9, or S4/S5/S6 file_version ≥ 2).")

In [19]:
# ── HRS2 Trial Viewer: Interactive Per-Trial Grid + Zoom ──────────────────────
from IPython.display import display as _disp

def _render_trv(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    tp = _apply_merge(trials)
    print(f'\n── Trial Viewer: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_hrs2_trials(
        tp, header,
        pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        emg_blocks=emg_blocks,
    )

_trv_widget, _trv_render = make_viewer(_all_recordings, _active_rec_label, _render_trv)
_disp(_trv_widget)
_trv_render()